# Clinical Trials Analysis
## Notebook 01 — Data Extraction & First Exploration

**Proyecto:** Data Wrangling — Individual Project  
**Fuente de datos:** ClinicalTrials.gov API v2 (pública, sin API key)  
**Objetivo del notebook:** Extraer datos, explorar su estructura y guardar el raw dataset.

---

## 0. Setup e imports

In [1]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn", "requests"])

NameError: name 'sys' is not defined

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Nuestro módulo de extracción
from src.api_extraction import fetch_all_trials, save_raw_data, quick_explore

print('✅ Imports correctos')

ModuleNotFoundError: No module named 'pandas'

## 1. Extracción de datos via API

La API de **ClinicalTrials.gov v2** es pública, sin registro ni API key.

- Base URL: `https://clinicaltrials.gov/api/v2/studies`
- Rate limit: ~50 requests/min
- Paginación: por tokens (hasta 1000 estudios/página)
- Más de 500.000 ensayos disponibles globalmente

In [ ]:
# Áreas terapéuticas principales — estas son las condiciones que consultaremos
CONDITIONS = [
    "cancer",
    "diabetes",
    "cardiovascular",
    "infectious disease",
    "neurological",
    "respiratory",
    "rare disease",
]

print(f'Áreas terapéuticas a extraer: {len(CONDITIONS)}')
for c in CONDITIONS:
    print(f'  • {c}')

In [ ]:
# ⏱️ Esta celda tarda ~5-10 minutos dependiendo de tu conexión
# Extrae hasta 7000 estudios en total

df_raw = fetch_all_trials(
    conditions=CONDITIONS,
    max_studies=7000,
    delay=1.2   # segundos entre peticiones (respeta el rate limit)
)

print(f'\nDataset extraído: {df_raw.shape}')

In [ ]:
# Guardamos los datos crudos
filepath = save_raw_data(df_raw, 'trials_raw.csv')
print(f'Archivo guardado: {filepath}')

## 2. Primera exploración del dataset

In [ ]:
# Vista general
df_raw.head()

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumnas:', list(df_raw.columns))
print('\nTipos de datos:')
print(df_raw.dtypes)

In [ ]:
# Valores nulos
print('Valores nulos por columna:')
nulls = df_raw.isnull().sum()
pct = (nulls / len(df_raw) * 100).round(1)
pd.DataFrame({'nulls': nulls, 'pct_%': pct})[nulls > 0]

In [ ]:
# Estadísticas descriptivas
df_raw.describe()

## 3. Exploración visual rápida

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploración inicial — Clinical Trials Dataset', fontsize=14, fontweight='bold')

# 1. Distribución por estado
status_counts = df_raw['status'].value_counts().head(8)
axes[0].barh(status_counts.index, status_counts.values, color='steelblue')
axes[0].set_title('Distribución por Estado')
axes[0].set_xlabel('Número de ensayos')

# 2. Distribución por fase
phase_counts = df_raw['phase'].value_counts()
axes[1].pie(phase_counts.values, labels=phase_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribución por Fase')

# 3. Top 10 países
countries_exp = df_raw['countries'].dropna().str.split('; ').explode()
top_countries = countries_exp[countries_exp != 'Unknown'].value_counts().head(10)
axes[2].barh(top_countries.index[::-1], top_countries.values[::-1], color='coral')
axes[2].set_title('Top 10 Países')
axes[2].set_xlabel('Número de ensayos')

plt.tight_layout()
plt.savefig('../data/raw/exploration_initial.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

In [ ]:
# Resumen ejecutivo del día 1
quick_explore(df_raw)

## 4. Conclusiones del Día 1

**Datos extraídos:**
- Fuente: ClinicalTrials.gov API v2
- Registros totales: _(completar tras ejecución)_
- Columnas: 15 variables por ensayo

**Observaciones preliminares:**
- _(despues de ver los gráficos lo completo)_

**mañana:**
- Ejecutar `main_cleaning()` del módulo `cleaning.py`
- Aplicar las 7 técnicas de limpieza
- Guardar dataset limpio en `/data/clean/`